# Phase 2A — LightGBM retune

**Why**: the initial LightGBM run in Phase 2A diagnostics had `best_iteration=1`
(pathological early stopping) and `random_normal_10` (F6D pure noise) in the
top 20 by gain. Both symptoms point at config issues, not a defect in the
underlying tree-boosting approach.

**Fix plan**:
1. Replace the 15% random validation split with the **temporal validation
   split from Test 3** (cohorts 202611-202612). Same distribution as OOT;
   stops early stopping from over-/under-firing on a non-representative slice.
2. **Optuna search** (30 trials, TPE sampler) over:
   `learning_rate`, `num_leaves`, `min_child_samples`, `reg_alpha`,
   `reg_lambda`, `feature_fraction`, `bagging_fraction`. Cap at
   `n_estimators=2000` with `early_stopping_rounds=100`.
3. **Verification gates** after refit:
   - `best_iteration` > 50
   - F6D pure-random features (`random_*`) absent from top-20 gain
   - OOT Gini in [0.50, 0.75] sanity range
4. **Calibration** via Platt + isotonic on the same val set (same recipe as
   Test 3, leak-free w.r.t. OOT).
5. **Save** tuned model + tuning trace + feature importance + comparison.

In [1]:
import sys, time, json, joblib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler

from src.governance import filter_pd_eligible, get_meta_columns
from src.evaluation import (
    compute_gini, compute_ks, compute_brier, compute_calibration_metrics,
)
from src.calibration import (
    make_calibration_split, fit_platt, fit_isotonic, apply_calibrator,
)

ABT_PATH     = REPO_ROOT / "artifacts/phase2_rerun_v2/modeling_abt.parquet"
CATALOG_PATH = REPO_ROOT / "artifacts/phase2_rerun_v2/modeling_feature_catalog.csv"
OUT_DIR      = REPO_ROOT / "artifacts/pd_model"
OUT_DIR.mkdir(parents=True, exist_ok=True)

T0 = time.time()
optuna.logging.set_verbosity(optuna.logging.WARNING)
catalog = pd.read_csv(CATALOG_PATH)
all_pd_eligible = filter_pd_eligible(catalog)
print(f"PD-eligible pool: {len(all_pd_eligible)}")

PD-eligible pool: 435


In [2]:
# Load only the columns we need
df = pd.read_parquet(ABT_PATH, columns=all_pd_eligible + get_meta_columns())
print(f"ABT loaded: {df.shape}")

# Temporal calibration split (same as Test 3)
TRAIN_FOR_MODEL = list(range(202509, 202611))
CALIB_PERIODS   = [202611, 202612]
df["split_new"] = make_calibration_split(df, TRAIN_FOR_MODEL, CALIB_PERIODS)
print("\nSplit counts:")
print(df["split_new"].value_counts().to_string())

# Cast categoricals for native handling
CAT_FEATURES = ["app_nom_branch", "app_nom_gender", "app_nom_job_code",
                "app_nom_marital_status", "app_nom_city", "app_nom_home_status"]
cat_in_pool = [c for c in CAT_FEATURES if c in all_pd_eligible]
for c in cat_in_pool:
    df[c] = df[c].astype("category")
print(f"\nCategorical features in pool: {cat_in_pool}")

mask_tr  = df["split_new"] == "train_for_model"
mask_val = df["split_new"] == "calib"
mask_oot = df["split_new"] == "oot"

X_tr  = df.loc[mask_tr,  all_pd_eligible]
y_tr  = df.loc[mask_tr,  "default_flag_12m"].astype(int).to_numpy()
X_val = df.loc[mask_val, all_pd_eligible]
y_val = df.loc[mask_val, "default_flag_12m"].astype(int).to_numpy()
X_oot = df.loc[mask_oot, all_pd_eligible]
y_oot = df.loc[mask_oot, "default_flag_12m"].astype(int).to_numpy()

n_pos, n_neg = int(y_tr.sum()), int((1 - y_tr).sum())
spw = n_neg / max(n_pos, 1)
print(f"\nTrain: {len(y_tr):,} (pos={n_pos}, neg={n_neg})")
print(f"Val  : {len(y_val):,} (pos={int(y_val.sum())}, neg={int((1-y_val).sum())})")
print(f"OOT  : {len(y_oot):,} (pos={int(y_oot.sum())}, neg={int((1-y_oot).sum())})")
print(f"scale_pos_weight = {spw:.3f}")

ABT loaded: (534314, 439)



Split counts:
split_new
train_for_model    340727
oot                144789
calib               48798

Categorical features in pool: ['app_nom_branch', 'app_nom_gender', 'app_nom_job_code', 'app_nom_marital_status', 'app_nom_city', 'app_nom_home_status']



Train: 340,727 (pos=5910, neg=334817)
Val  : 48,798 (pos=451, neg=48347)
OOT  : 144,789 (pos=1227, neg=143562)
scale_pos_weight = 56.653


## Optuna search (30 trials)

Objective: maximise validation AUC at the early-stopping-best iteration.
Search space chosen to span aggressive (deep, low reg) to conservative
(shallow, high reg) configurations.

In [3]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "scale_pos_weight": spw,
        "n_jobs": -1,
        "seed": 42,
        "learning_rate":      trial.suggest_float("learning_rate", 0.01, 0.10, log=True),
        "num_leaves":         trial.suggest_int("num_leaves", 15, 127),
        "min_child_samples":  trial.suggest_int("min_child_samples", 20, 1000, log=True),
        "reg_alpha":          trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda":         trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "feature_fraction":   trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction":   trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq":       1,
        "max_depth":          -1,
    }
    dtr  = lgb.Dataset(X_tr,  label=y_tr,  categorical_feature=cat_in_pool, free_raw_data=False)
    dval = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_in_pool,
                       reference=dtr, free_raw_data=False)
    model = lgb.train(
        params, dtr,
        num_boost_round=2000,
        valid_sets=[dval],
        valid_names=["val"],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
    )
    return model.best_score["val"]["auc"]

study = optuna.create_study(direction="maximize",
                             sampler=TPESampler(seed=42, n_startup_trials=8))
t0 = time.time()
study.optimize(objective, n_trials=30, show_progress_bar=False)
print(f"\nOptuna search wall: {time.time()-t0:.1f}s")
print(f"Best val AUC: {study.best_value:.5f}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k:<20} {v}")


Optuna search wall: 755.5s
Best val AUC: 0.86817
Best params:
  learning_rate        0.03912141628549695
  num_leaves           20
  min_child_samples    213
  reg_alpha            0.004809461967501573
  reg_lambda           0.0018205657658407262
  feature_fraction     0.9744427686266666
  bagging_fraction     0.9828160165372797


## Refit best parameters

Train with the best hyperparameters and capture the converged `best_iteration`.

In [4]:
best_params = dict(study.best_params)
best_params.update({
    "objective": "binary",
    "metric": "auc",
    "verbosity": -1,
    "boosting_type": "gbdt",
    "scale_pos_weight": spw,
    "n_jobs": -1,
    "seed": 42,
    "max_depth": -1,
    "bagging_freq": 1,
})

dtr  = lgb.Dataset(X_tr,  label=y_tr,  categorical_feature=cat_in_pool, free_raw_data=False)
dval = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_in_pool,
                   reference=dtr, free_raw_data=False)
t0 = time.time()
booster = lgb.train(
    best_params, dtr,
    num_boost_round=2000,
    valid_sets=[dval],
    valid_names=["val"],
    callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
)
print(f"Refit wall: {time.time()-t0:.1f}s")
print(f"best_iteration: {booster.best_iteration}")
print(f"best_score val AUC: {booster.best_score['val']['auc']:.5f}")

Refit wall: 18.6s
best_iteration: 141
best_score val AUC: 0.86817


In [5]:
# Score everything (train, val, oot)
df["pd_lgb_raw"] = booster.predict(df[all_pd_eligible], num_iteration=booster.best_iteration)

# Calibrate on val (NOT OOT)
s_val = df.loc[mask_val, "pd_lgb_raw"].to_numpy()
y_val_arr = y_val
platt = fit_platt(s_val, y_val_arr)
iso   = fit_isotonic(s_val, y_val_arr)
df["pd_lgb_platt"] = apply_calibrator(df["pd_lgb_raw"].to_numpy(), platt, "platt")
df["pd_lgb_iso"]   = apply_calibrator(df["pd_lgb_raw"].to_numpy(), iso,   "isotonic")

# OOT metrics
def metrics(y, s):
    cal = compute_calibration_metrics(y, s)
    return {
        "auc"      : (compute_gini(y, s) + 1) / 2,
        "gini"     : compute_gini(y, s),
        "ks"       : compute_ks(y, s),
        "brier"    : compute_brier(y, s),
        "ece"      : cal.get("ece", float("nan")),
        "mean_pred": cal.get("mean_pred", float("nan")),
        "base_rate": cal.get("base_rate", float("nan")),
    }

oot_results = {
    "pd_lgb_raw"  : metrics(y_oot, df.loc[mask_oot, "pd_lgb_raw"].to_numpy()),
    "pd_lgb_platt": metrics(y_oot, df.loc[mask_oot, "pd_lgb_platt"].to_numpy()),
    "pd_lgb_iso"  : metrics(y_oot, df.loc[mask_oot, "pd_lgb_iso"].to_numpy()),
}
print("=== Tuned LightGBM OOT metrics ===")
print(pd.DataFrame(oot_results).round(5).to_string())

=== Tuned LightGBM OOT metrics ===
           pd_lgb_raw  pd_lgb_platt  pd_lgb_iso
auc           0.89754       0.89754     0.89625
gini          0.79508       0.79508     0.79250
ks            0.63799       0.63799     0.63436
brier         0.14355       0.00788     0.00783
ece           0.28861       0.00124     0.00091
mean_pred     0.29709       0.00939     0.00931
base_rate     0.00847       0.00847     0.00847


## Verification gates

- `best_iteration` > 50
- No F6D (`random_*`) feature in top-20 gain
- OOT Gini in [0.50, 0.75]

In [6]:
# Feature importance
imp_split = booster.feature_importance(importance_type="split")
imp_gain  = booster.feature_importance(importance_type="gain")
imp_df = pd.DataFrame({
    "feature": all_pd_eligible,
    "importance_split": imp_split,
    "importance_gain":  imp_gain,
}).sort_values("importance_gain", ascending=False).reset_index(drop=True)

top20 = imp_df.head(20)
print("Top 20 LightGBM (tuned) features by gain:")
print(top20.to_string(index=False))

f6d_in_top20 = [f for f in top20["feature"] if f.startswith("random_")]
print(f"\nF6D in top-20 gain: {len(f6d_in_top20)}  {f6d_in_top20}")

# Also show "F6D anywhere" to confirm they are properly suppressed
f6d_all = imp_df[imp_df["feature"].str.startswith("random_")]
f6d_any = f6d_all[f6d_all["importance_gain"] > 0]
print(f"F6D with non-zero gain anywhere: {len(f6d_any)} of {len(f6d_all)}")
if len(f6d_any) > 0:
    print(f6d_any.head(10).to_string(index=False))

Top 20 LightGBM (tuned) features by gain:
                      feature  importance_split  importance_gain
            job_code_x_income                93     1.392351e+06
       app_nom_marital_status                37     7.020252e+05
             age_x_n_children                63     4.260934e+05
               app_nom_gender                91     3.364310e+05
    synth_int_inc_x_nchildren                30     2.938565e+05
         synth_thin_file_flag                49     2.830825e+05
                act_age_noisy                24     2.633465e+05
    synth_household_income_v1                 5     6.962889e+04
         mean_age_by_job_code                31     6.858058e+04
                      act_age                29     6.067845e+04
mean_income_by_marital_status                42     4.972123e+04
  synth_int_spend_x_nchildren                29     4.760228e+04
         marital_x_n_children                32     3.849148e+04
      synth_employment_months                 7 

In [7]:
gates = {
    "best_iteration_gt_50":  booster.best_iteration > 50,
    "f6d_in_top20_zero":     len(f6d_in_top20) == 0,
    "oot_gini_in_range":     0.50 <= oot_results["pd_lgb_raw"]["gini"] <= 0.75,
}
print("Verification gates:")
for k, v in gates.items():
    print(f"  [{('PASS' if v else 'FAIL')}] {k}")

all_pass = all(gates.values())
print(f"\nOverall: {'ALL PASS' if all_pass else 'SOME FAILED — investigate'}")

Verification gates:
  [PASS] best_iteration_gt_50
  [PASS] f6d_in_top20_zero
  [FAIL] oot_gini_in_range

Overall: SOME FAILED — investigate


## Save artifacts

In [8]:
# Save model
joblib.dump({
    "booster": booster,
    "best_params": best_params,
    "best_iteration": int(booster.best_iteration),
    "best_val_auc": float(booster.best_score["val"]["auc"]),
    "scale_pos_weight": float(spw),
    "categorical_features": cat_in_pool,
    "platt": platt,
    "isotonic": iso,
    "feature_list": all_pd_eligible,
    "train_for_model_periods": TRAIN_FOR_MODEL,
    "calib_periods": CALIB_PERIODS,
}, OUT_DIR / "lightgbm_tuned_model.pkl")
print(f"Saved: {OUT_DIR / 'lightgbm_tuned_model.pkl'}")

# Save feature importance
imp_df.to_csv(OUT_DIR / "lightgbm_feature_importance.csv", index=False)
print(f"Saved: {OUT_DIR / 'lightgbm_feature_importance.csv'}")

# Save tuning results
trial_records = []
for t in study.trials:
    trial_records.append({
        "trial": t.number,
        "value": t.value,
        "params": t.params,
        "state": str(t.state),
    })
tuning_out = {
    "n_trials": len(study.trials),
    "best_trial": study.best_trial.number,
    "best_value": float(study.best_value),
    "best_params": study.best_params,
    "post_refit_best_iteration": int(booster.best_iteration),
    "post_refit_best_val_auc":   float(booster.best_score["val"]["auc"]),
    "scale_pos_weight": float(spw),
    "splits": {
        "train_for_model_periods": TRAIN_FOR_MODEL,
        "calib_periods": CALIB_PERIODS,
        "n_train": int(mask_tr.sum()),
        "n_val":   int(mask_val.sum()),
        "n_oot":   int(mask_oot.sum()),
    },
    "oot_metrics": oot_results,
    "verification_gates": gates,
    "f6d_in_top20": list(f6d_in_top20),
    "trials": trial_records,
}
with open(OUT_DIR / "lightgbm_tuning_results.json", "w") as f:
    json.dump(tuning_out, f, indent=2, default=str)
print(f"Saved: {OUT_DIR / 'lightgbm_tuning_results.json'}")

Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\pd_model\lightgbm_tuned_model.pkl
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\pd_model\lightgbm_feature_importance.csv
Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\pd_model\lightgbm_tuning_results.json


## Comparison with LR variants and pre-retune LightGBM

In [9]:
# Pre-retune LightGBM metrics (from diagnostics test 4)
with open(REPO_ROOT / "artifacts/phase2_rerun_v2/diagnostics/test4_lightgbm.json") as f:
    pre_retune = json.load(f)

# LR variants (full-F6E from initial Phase 2A; no-F6E from Test 1)
with open(REPO_ROOT / "artifacts/phase2_rerun_v2/model_metrics.json") as f:
    lr_full = json.load(f)
with open(REPO_ROOT / "artifacts/phase2_rerun_v2/diagnostics/test1_f6e_ablation.json") as f:
    lr_nof6e = json.load(f)

cmp = pd.DataFrame({
    "LR_full_F6E_oot":  lr_full["oot"],
    "LR_no_F6E_oot":    lr_nof6e["metrics"]["oot"],
    "LGB_pre_retune_raw_oot":   pre_retune["oot_metrics"]["pd_lgb_raw"],
    "LGB_pre_retune_platt_oot": pre_retune["oot_metrics"]["pd_lgb_platt"],
    "LGB_tuned_raw_oot":        oot_results["pd_lgb_raw"],
    "LGB_tuned_platt_oot":      oot_results["pd_lgb_platt"],
})
print("=== Side-by-side OOT comparison ===")
print(cmp.round(5).to_string())

# Best iteration comparison
print(f"\nbest_iteration: pre-retune=1, post-retune={booster.best_iteration}")
print(f"F6D in top-20 gain: pre-retune=1, post-retune={len(f6d_in_top20)}")

# Top-7 feature comparison: LR no-F6E uses 7 features
LR_NO_F6E = lr_nof6e["final_features"]
print(f"\nLR (no-F6E) final 7 features:")
for f in LR_NO_F6E:
    rank = imp_df[imp_df['feature'] == f].index
    rank_str = f"LGB rank {int(rank[0])+1}" if len(rank) > 0 else "(not in pool)"
    print(f"  {f:<35}  {rank_str}")

print(f"\nLightGBM (tuned) top-7 by gain:")
for i, row in imp_df.head(7).iterrows():
    print(f"  {row['feature']:<35}  gain={row['importance_gain']:>10.0f}  splits={row['importance_split']}")

print(f"\nTotal Phase 2A LGB retune wall: {time.time() - T0:.1f}s")

=== Side-by-side OOT comparison ===
           LR_full_F6E_oot  LR_no_F6E_oot  LGB_pre_retune_raw_oot  LGB_pre_retune_platt_oot  LGB_tuned_raw_oot  LGB_tuned_platt_oot
gini               0.71812        0.67226                 0.74525                   0.74525            0.79508              0.79508
ks                 0.54755        0.53948                 0.59611                   0.59611            0.63799              0.63799
brier              0.00816        0.00821                 0.01099                   0.00810            0.14355              0.00788
auc                0.85906        0.83613                 0.87262                   0.87262            0.89754              0.89754
ece                    NaN            NaN                 0.04492                   0.00095            0.28861              0.00124
mean_pred              NaN            NaN                 0.05339                   0.00938            0.29709              0.00939
base_rate              NaN            Na